# MLOps Experiment Tracking & Model Registry

Systematic machine learning development requires capturing the full provenance of training runs:
1. **Experiment Tracking**: Logging hyperparameters, metric trajectories over epochs, environment tags, and code versions.
2. **Comparative Analysis**: Querying runs across hyperparameter grids to identify Pareto-optimal models.
3. **Artifact Logging**: Persisting trained weights, configuration files, and evaluation confusion matrices.
4. **Model Registry**: Formalizing the lifecycle stages of trained models (`None` -> `Staging` -> `Production` -> `Archived`).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('07-mlops/experiment-tracking/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import tracker
print('tracker loaded')

## 1. Tracking Hyperparameters and Metric Trajectories
Tracking allows teams to reproduce any model trained across team members and compute clusters.

In [ ]:
from tracker import ExperimentTracker

tracker = ExperimentTracker()

# Run 1: LightGBM baseline
run1 = tracker.start_run(experiment_name="customer_churn", run_name="lgbm_depth4")
tracker.log_params({"model_type": "LightGBM", "max_depth": 4, "learning_rate": 0.05, "n_estimators": 100})

for epoch in range(1, 5):
    train_loss = 0.60 / epoch
    val_auc = 0.75 + (epoch * 0.03)
    tracker.log_metric("loss", train_loss, step=epoch)
    tracker.log_metric("val_auc", val_auc, step=epoch)

tracker.log_artifact("model_artifact", {"type": "booster", "features": ["tenure", "monthly_charges"]})
tracker.end_run()

# Run 2: Deeper model with regularization
run2 = tracker.start_run(experiment_name="customer_churn", run_name="lgbm_depth8_reg")
tracker.log_params({"model_type": "LightGBM", "max_depth": 8, "learning_rate": 0.02, "n_estimators": 150})

for epoch in range(1, 5):
    train_loss = 0.50 / epoch
    val_auc = 0.78 + (epoch * 0.035)
    tracker.log_metric("loss", train_loss, step=epoch)
    tracker.log_metric("val_auc", val_auc, step=epoch)

tracker.log_artifact("model_artifact", {"type": "booster", "features": ["tenure", "monthly_charges"]})
tracker.end_run()

print(f"Logged 2 runs in experiment 'customer_churn'.")

## 2. Querying and Comparing Experiment Runs
Ranking runs by validation metrics to select the deployment candidate.

In [ ]:
# Search runs ordered by validation AUC
runs = tracker.search_runs(experiment_name="customer_churn", order_by_metric="val_auc", descending=True)

print("--- Leaderboard ---")
for r in runs:
    metrics = r.get_latest_metrics()
    print(f"Run: {r.run_name:18s} | AUC: {metrics['val_auc']:.4f} | Loss: {metrics['loss']:.4f} | Params: {r.params}")

# Structured comparison
comparison = tracker.compare_runs([r.run_id for r in runs])
best_run = runs[0]
print(f"\nBest candidate selected: {best_run.run_name} (ID: {best_run.run_id})")

## 3. Model Registry & Stage Transitions (MLflow Pattern)
Governing models through Staging, Production, and Automated Deprecation/Archival.

In [ ]:
from tracker import ModelRegistry

registry = ModelRegistry()

# 1. Register candidate from run 1
v1 = registry.register_model(
    model_name="churn_classifier",
    run_id=runs[1].run_id,
    artifact_name="model_artifact",
    description="Initial baseline model"
)
print(f"Registered {v1.model_name} v{v1.version} in stage '{v1.stage}'")

# Promote v1 to Production
registry.transition_stage("churn_classifier", version=1, target_stage="Production")
print(f"Promoted v1 to: {v1.stage}")

# 2. Register improved candidate from run 2
v2 = registry.register_model(
    model_name="churn_classifier",
    run_id=runs[0].run_id,
    artifact_name="model_artifact",
    description="High AUC regularized model"
)
print(f"Registered {v2.model_name} v{v2.version} in stage '{v2.stage}'")

# Promote v2 to Production (automatically archiving v1)
registry.transition_stage("churn_classifier", version=2, target_stage="Production", archive_existing_versions=True)
print(f"After promotion:")
print(f" - v1 stage: {v1.stage}")
print(f" - v2 stage: {v2.stage}")

current_prod = registry.get_production_model("churn_classifier")
print(f"Active Production Model: v{current_prod.version} (Source Run: {current_prod.run_id})")